In [ ]:
#| default_exp aws

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore.all import *
import pulumi
import pulumi_aws as aws
from typing import Optional, List, Dict, Any
from pathlib import Path

## Storage

S3 bucket with security best practices by default

In [ ]:
#| export
@delegates(aws.s3.Bucket.__init__)
def storage(name:str=None, # Bucket name (auto-generated if None)
            versioning:bool=True, # Enable versioning
            encryption:bool=True, # Enable encryption
            public:bool=False, # Allow public access
            **kwargs):
    "Create S3 bucket with security defaults"
    bucket_name = name or f"pullup-{pulumi.get_project()}-{pulumi.get_stack()}"
    
    # Create bucket
    bucket = aws.s3.Bucket(bucket_name, **kwargs)
    
    # Block public access by default
    if not public:
        aws.s3.BucketPublicAccessBlock(f"{bucket_name}-public-block",
            bucket=bucket.id,
            block_public_acls=True,
            block_public_policy=True,
            ignore_public_acls=True,
            restrict_public_buckets=True)
    
    # Enable versioning
    if versioning:
        aws.s3.BucketVersioningV2(f"{bucket_name}-versioning",
            bucket=bucket.id,
            versioning_configuration=aws.s3.BucketVersioningV2VersioningConfigurationArgs(
                status="Enabled"))
    
    # Enable server-side encryption
    if encryption:
        aws.s3.BucketServerSideEncryptionConfigurationV2(f"{bucket_name}-encryption",
            bucket=bucket.id,
            rules=[aws.s3.BucketServerSideEncryptionConfigurationV2RuleArgs(
                apply_server_side_encryption_by_default=aws.s3.BucketServerSideEncryptionConfigurationV2RuleApplyServerSideEncryptionByDefaultArgs(
                    sse_algorithm="AES256"))])
    
    return bucket

## VPC

VPC with public and private subnets

In [ ]:
#| export
@delegates(aws.ec2.Vpc.__init__)
def vpc(name:str=None, # VPC name
        cidr:str="10.0.0.0/16", # CIDR block
        azs:int=2, # Number of availability zones
        nat:bool=True, # Enable NAT gateway
        **kwargs):
    "Create VPC with public and private subnets"
    vpc_name = name or f"pullup-{pulumi.get_project()}"
    
    # Create VPC
    main_vpc = aws.ec2.Vpc(vpc_name,
        cidr_block=cidr,
        enable_dns_hostnames=True,
        enable_dns_support=True,
        tags={"Name": vpc_name},
        **kwargs)
    
    # Internet Gateway
    igw = aws.ec2.InternetGateway(f"{vpc_name}-igw",
        vpc_id=main_vpc.id,
        tags={"Name": f"{vpc_name}-igw"})
    
    # Get availability zones
    available_azs = aws.get_availability_zones(state="available")
    
    public_subnets = []
    private_subnets = []
    
    # Create subnets
    for i in range(min(azs, len(available_azs.names))):
        az = available_azs.names[i]
        
        # Public subnet
        public_subnet = aws.ec2.Subnet(f"{vpc_name}-public-{i}",
            vpc_id=main_vpc.id,
            cidr_block=f"10.0.{i}.0/24",
            availability_zone=az,
            map_public_ip_on_launch=True,
            tags={"Name": f"{vpc_name}-public-{i}"})
        public_subnets.append(public_subnet)
        
        # Private subnet
        private_subnet = aws.ec2.Subnet(f"{vpc_name}-private-{i}",
            vpc_id=main_vpc.id,
            cidr_block=f"10.0.{100+i}.0/24",
            availability_zone=az,
            tags={"Name": f"{vpc_name}-private-{i}"})
        private_subnets.append(private_subnet)
    
    # Public route table
    public_rt = aws.ec2.RouteTable(f"{vpc_name}-public-rt",
        vpc_id=main_vpc.id,
        routes=[aws.ec2.RouteTableRouteArgs(
            cidr_block="0.0.0.0/0",
            gateway_id=igw.id)],
        tags={"Name": f"{vpc_name}-public-rt"})
    
    # Associate public subnets with public route table
    for i, subnet in enumerate(public_subnets):
        aws.ec2.RouteTableAssociation(f"{vpc_name}-public-rta-{i}",
            subnet_id=subnet.id,
            route_table_id=public_rt.id)
    
    # NAT Gateway (if enabled)
    if nat:
        eip = aws.ec2.Eip(f"{vpc_name}-nat-eip", domain="vpc")
        nat_gateway = aws.ec2.NatGateway(f"{vpc_name}-nat",
            subnet_id=public_subnets[0].id,
            allocation_id=eip.id,
            tags={"Name": f"{vpc_name}-nat"})
        
        # Private route table
        private_rt = aws.ec2.RouteTable(f"{vpc_name}-private-rt",
            vpc_id=main_vpc.id,
            routes=[aws.ec2.RouteTableRouteArgs(
                cidr_block="0.0.0.0/0",
                nat_gateway_id=nat_gateway.id)],
            tags={"Name": f"{vpc_name}-private-rt"})
        
        # Associate private subnets with private route table
        for i, subnet in enumerate(private_subnets):
            aws.ec2.RouteTableAssociation(f"{vpc_name}-private-rta-{i}",
                subnet_id=subnet.id,
                route_table_id=private_rt.id)
    
    return dict(vpc=main_vpc, public_subnets=public_subnets, 
                 private_subnets=private_subnets, igw=igw)

## Fargate

ECS Fargate cluster with Docker support

In [ ]:
#| export
class Fargate:
    "ECS Fargate deployment with Docker build support"
    def __init__(self, 
                 app_dir:Path=None, # Application directory with Dockerfile
                 name:str=None, # Service name
                 vpc_config:dict=None, # VPC configuration from vpc()
                 image:str=None, # Docker image (if not building)
                 port:int=80, # Container port
                 cpu:int=256, # CPU units
                 memory:int=512, # Memory in MB
                 desired_count:int=1): # Number of tasks
        self.app_dir = Path(app_dir) if app_dir else None
        self.name = name or f"pullup-{pulumi.get_project()}"
        self.vpc_config = vpc_config or vpc()
        self.image = image
        self.port = port
        self.cpu = cpu
        self.memory = memory
        self.desired_count = desired_count
        self._cluster = None
        self._service = None
        
        # Validate Docker setup if building
        if app_dir and not image:
            dockerfile = self.app_dir / "Dockerfile"
            if not dockerfile.exists():
                raise ValueError(f"Dockerfile not found in {app_dir}")
    
    def deploy(self):
        "Deploy the Fargate service"
        # Create ECS cluster
        cluster = aws.ecs.Cluster(f"{self.name}-cluster",
            tags={"Name": f"{self.name}-cluster"})
        self._cluster = cluster
        
        # Create ECR repository if building from app_dir
        if self.app_dir and not self.image:
            repo = aws.ecr.Repository(f"{self.name}-repo",
                image_scanning_configuration=aws.ecr.RepositoryImageScanningConfigurationArgs(
                    scan_on_push=True),
                encryption_configurations=[aws.ecr.RepositoryEncryptionConfigurationArgs(
                    encryption_type="AES256")])
            
            # Build and push Docker image
            image = aws.ecr.get_authorization_token_output()
            import pulumi_docker as docker
            
            docker_image = docker.Image(f"{self.name}-image",
                build=docker.DockerBuildArgs(
                    context=str(self.app_dir),
                    dockerfile=str(self.app_dir / "Dockerfile"),
                    platform="linux/amd64"),
                image_name=repo.repository_url,
                registry=docker.RegistryArgs(
                    server=repo.repository_url,
                    username=image.user_name,
                    password=image.password))
            
            container_image = docker_image.image_name
        else:
            container_image = self.image
        
        # Create IAM role for task execution
        exec_role = aws.iam.Role(f"{self.name}-exec-role",
            assume_role_policy="""{
                "Version": "2012-10-17",
                "Statement": [{
                    "Effect": "Allow",
                    "Principal": {"Service": "ecs-tasks.amazonaws.com"},
                    "Action": "sts:AssumeRole"
                }]
            }""")
        
        aws.iam.RolePolicyAttachment(f"{self.name}-exec-policy",
            role=exec_role.name,
            policy_arn="arn:aws:iam::aws:policy/service-role/AmazonECSTaskExecutionRolePolicy")
        
        # Create task definition
        task_def = aws.ecs.TaskDefinition(f"{self.name}-task",
            family=self.name,
            cpu=str(self.cpu),
            memory=str(self.memory),
            network_mode="awsvpc",
            requires_compatibilities=["FARGATE"],
            execution_role_arn=exec_role.arn,
            container_definitions=pulumi.Output.json_dumps([{
                "name": self.name,
                "image": container_image,
                "portMappings": [{"containerPort": self.port}],
                "logConfiguration": {
                    "logDriver": "awslogs",
                    "options": {
                        "awslogs-group": f"/ecs/{self.name}",
                        "awslogs-region": aws.get_region().name,
                        "awslogs-stream-prefix": "ecs"
                    }
                }
            }]))
        
        # Create CloudWatch log group
        aws.cloudwatch.LogGroup(f"{self.name}-logs",
            name=f"/ecs/{self.name}",
            retention_in_days=7)
        
        # Create security group
        sg = aws.ec2.SecurityGroup(f"{self.name}-sg",
            vpc_id=self.vpc_config.vpc.id,
            ingress=[aws.ec2.SecurityGroupIngressArgs(
                protocol="tcp",
                from_port=self.port,
                to_port=self.port,
                cidr_blocks=["0.0.0.0/0"])],
            egress=[aws.ec2.SecurityGroupEgressArgs(
                protocol="-1",
                from_port=0,
                to_port=0,
                cidr_blocks=["0.0.0.0/0"])])
        
        # Create ECS service
        service = aws.ecs.Service(f"{self.name}-service",
            cluster=cluster.arn,
            desired_count=self.desired_count,
            launch_type="FARGATE",
            task_definition=task_def.arn,
            network_configuration=aws.ecs.ServiceNetworkConfigurationArgs(
                assign_public_ip=True,
                subnets=[s.id for s in self.vpc_config.public_subnets],
                security_groups=[sg.id]))
        
        self._service = service
        return self

In [ ]:
#| export
@delegates(Fargate.__init__)
def fargate(**kwargs):
    "Create Fargate deployment (call .deploy() to deploy)"
    return Fargate(**kwargs)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()